In [11]:
from groq import Groq
import os
import requests
from dotenv import load_dotenv
load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [12]:


def jina_embedding_model(input_data) :
    url = "https://api.jina.ai/v1/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.getenv("JINA_API_KEY")}"
    }
    data = {
        "model": "jina-embeddings-v2-base-en",
        "task": "separation",
        "input": input_data
    }
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    return response
    

In [13]:
from pinecone import Pinecone, ServerlessSpec
EMBED_DIM = 768
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
INDEX_NAME = "jina-try-2"

# Create the index if it doesn't already exist
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(INDEX_NAME)







In [14]:
def retrieve_context(query, top_k=3, namespace="my-namespace"):
    """
    Embeds the query, retrieves top_k matches from Pinecone,
    and formats them into a single context string with page numbers.
    """
    query_embedding = jina_embedding_model([query]).json()['data'][0]['embedding']

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True
    )

    context_parts = []
    for i, match in enumerate(results.matches, start=1):
        text = match.metadata.get('text', '')
        page = match.metadata.get('page_number', 'N/A')
        score = match.score
        context_parts.append(
            f"[Chunk {i} | Page {page} | Relevance: {score:.4f}]\n{text}"
        )

    context = "\n\n---\n\n".join(context_parts)
    return context, results.matches

In [15]:
def generate_answer(query, top_k=3, namespace="my-namespace", model="llama-3.3-70b-versatile"):
    """
    Retrieves relevant context from Pinecone and asks Groq's LLM
    to answer the query using ONLY that context.
    """
    context, matches = retrieve_context(query, top_k=top_k, namespace=namespace)

    system_prompt = (
        "You are a helpful assistant that answers questions using ONLY the "
        "provided context. If the answer is not contained in the context, "
        "say you don't have enough information to answer, rather than guessing. "
        "When relevant, mention which page(s) the information came from."
    )

    user_prompt = f"""Context:
{context}

Question: {query}

Answer the question clearly and concisely based on the context above."""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=1000
    )

    answer = response.choices[0].message.content
    return answer, context, matches

In [16]:
query = "when i open my computer i see a blue screen of death also tell me how i should fix this issue"

answer, context, matches = generate_answer(query, top_k=3)

print("========================================")
print("QUESTION:", query)
print("========================================")
print("\nANSWER:\n")
print(answer)
print("\n" + "========================================")
print("SOURCES USED:")
for i, match in enumerate(matches, start=1):
    page = match.metadata.get('page_number', 'N/A')
    print(f"  Chunk {i}: Page {page} (score: {match.score:.4f})")

QUESTION: when i open my computer i see a blue screen of death also tell me how i should fix this issue

ANSWER:

According to the context (Page 1), a blue screen of death (BSOD) can indicate hardware or driver issues. To troubleshoot this issue, you may need to update drivers, check for hardware problems, or scan for malware. 

It's also recommended to follow some general troubleshooting steps mentioned in the context (Page 3), such as performing basic checks, researching the problem, reviewing recent changes, using safe mode, and checking hardware components. Additionally, updating drivers and software (Page 3) can help resolve compatibility issues and instability.

If you're unsure about how to proceed, you can use diagnostic tools (Page 4) or check the event viewer (Page 4) for error messages. If the issue persists, it's best to seek professional assistance to avoid causing further damage.

SOURCES USED:
  Chunk 1: Page 1 (score: 0.8247)
  Chunk 2: Page 3 (score: 0.8068)
  Chunk 3: